# Separating Output Feedback Controller for Fault Diagnosis

## Overview

This notebook demonstrates **output feedback control** for active fault diagnosis.

### Key Concept

Instead of optimizing a single constant input, we design a **time-varying feedback controller**:

```
u[t] = u_nom[t] + K[t] @ (y[t] - y_nom[t])
```

where:
- `u_nom[t]`: Nominal feedforward control (tracks desired trajectory)
- `K[t]`: Feedback gain matrix (optimized for fault separation)
- `y[t]`: Actual observation
- `y_nom[t]`: Nominal observation

### Dual Objectives

1. **Track nominal trajectory** (via feedforward `u_nom`)
2. **Maximize fault separation** (via optimized feedback `K`)

### Comparison with Open-Loop

| Approach | Form | Advantages | Limitations |
|----------|------|------------|-------------|
| **Open-Loop** | `u = constant` | Simple, easy to optimize | Can't adapt to disturbances |
| **Output Feedback** | `u[t] = u_nom[t] + K[t]@Δy[t]` | Adaptive, tracks trajectory | More complex, higher dimensional |

In [ ]:
# Imports
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.patches import Rectangle, FancyArrowPatch
import jax.numpy as jnp

from go2_separating_feedback_controller import (
    SeparatingFeedbackOptimizer,
    create_scenarios,
    Interval
)

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 11

print("✓ Libraries imported successfully")

## 1. Create Nominal Trajectory

In [ ]:
# Synthetic circular trajectory
N = 20
dt = 0.5
t = np.linspace(0, (N-1)*dt, N)

# Circle parameters
radius = 2.0
omega_nom = 0.3

# Nominal trajectory
x_nom = radius * np.cos(omega_nom * t)
y_nom = radius * np.sin(omega_nom * t)
theta_nom = omega_nom * t + np.pi/2

x_nom_traj = np.column_stack([x_nom, y_nom, theta_nom])

# Nominal control
vx_nom = -radius * omega_nom * np.sin(omega_nom * t)
vy_nom = radius * omega_nom * np.cos(omega_nom * t)
omega_cmd = np.ones(N) * omega_nom

u_nom_traj = np.column_stack([vx_nom, vy_nom, omega_cmd])

# Observation matrix (identity for simplicity)
C = np.eye(3)

print(f"Nominal trajectory created:")
print(f"  Length: {N} steps")
print(f"  Duration: {N*dt}s")
print(f"  Circle radius: {radius}m")
print(f"  Angular velocity: {omega_nom} rad/s")

# Visualize nominal trajectory
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Position trajectory
ax = axes[0]
ax.plot(x_nom, y_nom, 'b-o', linewidth=2, markersize=5, label='Nominal path')
ax.plot(x_nom[0], y_nom[0], 'go', markersize=12, label='Start')
ax.plot(x_nom[-1], y_nom[-1], 'ro', markersize=12, label='End')
ax.set_xlabel('X Position (m)', fontweight='bold')
ax.set_ylabel('Y Position (m)', fontweight='bold')
ax.set_title('Nominal Trajectory (XY)', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
ax.axis('equal')

# Control inputs
ax = axes[1]
ax.plot(t, vx_nom, 'b-', linewidth=2, label='vx')
ax.plot(t, vy_nom, 'r-', linewidth=2, label='vy')
ax.plot(t, omega_cmd, 'g-', linewidth=2, label='ω')
ax.set_xlabel('Time (s)', fontweight='bold')
ax.set_ylabel('Control Input', fontweight='bold')
ax.set_title('Nominal Control Inputs', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 2. Define Fault Scenarios

In [ ]:
scenarios = create_scenarios()

print("Fault Scenarios:")
print("="*60)
for i, s in enumerate(scenarios, 1):
    print(f"\n{i}. {s.name}")
    print(f"   Actuator (α): [{s.alpha_range[0]:.2f}, {s.alpha_range[1]:.2f}]")
    print(f"   Sensor scale: [{s.scale_range[0]:.2f}, {s.scale_range[1]:.2f}]")
    print(f"   Sensor bias:  [{np.degrees(s.bias_range[0]):.1f}°, "
          f"{np.degrees(s.bias_range[1]):.1f}°]")
print("="*60)

## 3. Setup Initial Conditions

In [ ]:
# Initial state uncertainty
x0_int = Interval(
    lower=jnp.array([-0.05, -0.05, -0.02]),
    upper=jnp.array([0.05, 0.05, 0.02])
)

print("Initial state uncertainty:")
print(f"  px ∈ [{x0_int.lower[0]:.3f}, {x0_int.upper[0]:.3f}] m")
print(f"  py ∈ [{x0_int.lower[1]:.3f}, {x0_int.upper[1]:.3f}] m")
print(f"  θ  ∈ [{x0_int.lower[2]:.3f}, {x0_int.upper[2]:.3f}] rad")

## 4. Optimize Feedback Controller

We optimize the feedback gains `K[t]` to minimize:

```
Loss = λ_sep * (overlap between scenarios) + λ_track * ||K||²
```

Where:
- `λ_sep = 1.0`: Weight on fault separation
- `λ_track = 0.01`: Small weight to keep gains small (stay close to nominal)

In [ ]:
# Create optimizer
optimizer = SeparatingFeedbackOptimizer(
    x_nom_traj=x_nom_traj,
    u_nom_traj=u_nom_traj,
    C=C,
    scenarios=scenarios,
    x0_int=x0_int,
    dt=dt,
    lambda_sep=1.0,
    lambda_track=0.01,
    obs_uncertainty=0.05
)

print("\nRunning optimization...")
K_opt, loss_opt = optimizer.optimize(
    learning_rate=0.001,
    num_iters=100,
    verbose=True
)

print(f"\n✓ Optimization complete!")
print(f"  Final loss: {loss_opt:.6f}")

## 5. Evaluate Results

In [ ]:
stats = optimizer.evaluate(K_opt)

print("\n" + "="*60)
print("RESULTS")
print("="*60)

print(f"\nSeparation Metrics:")
print(f"  Total overlap: {stats['total_overlap']:.6f} m²")

print(f"\nPairwise Overlaps:")
for key, val in stats['overlaps'].items():
    print(f"  {key:35s}: {val:.6f} m²")

print(f"\nFinal Position Intervals:")
for name, pos_int in stats['position_intervals'].items():
    width_x = pos_int.upper[0] - pos_int.lower[0]
    width_y = pos_int.upper[1] - pos_int.lower[1]
    print(f"  {name}:")
    print(f"    px ∈ [{pos_int.lower[0]:+.4f}, {pos_int.upper[0]:+.4f}] m  (width: {width_x:.4f})")
    print(f"    py ∈ [{pos_int.lower[1]:+.4f}, {pos_int.upper[1]:+.4f}] m  (width: {width_y:.4f})")

print(f"\nFeedback Gain Statistics:")
print(f"  Shape: {K_opt.shape}")
print(f"  Mean:  {np.mean(K_opt):.6f}")
print(f"  Std:   {np.std(K_opt):.6f}")
print(f"  Max:   {np.max(np.abs(K_opt)):.6f}")
print(f"  L2 norm: {stats['tracking_error']:.6f}")

## 6. Visualize Reachable Sets

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

colors = ['#3498db', '#e74c3c', '#2ecc71']  # Blue, Red, Green
alphas = [0.4, 0.4, 0.4]

# Left: Final reachable sets
ax = axes[0]

for i, (name, pos_int) in enumerate(stats['position_intervals'].items()):
    width = pos_int.upper[0] - pos_int.lower[0]
    height = pos_int.upper[1] - pos_int.lower[1]
    
    rect = Rectangle(
        (pos_int.lower[0], pos_int.lower[1]),
        width, height,
        linewidth=3,
        edgecolor=colors[i],
        facecolor=colors[i],
        alpha=alphas[i],
        label=name
    )
    ax.add_patch(rect)
    
    # Center point
    center_x = (pos_int.lower[0] + pos_int.upper[0]) / 2
    center_y = (pos_int.lower[1] + pos_int.upper[1]) / 2
    ax.plot(center_x, center_y, 'o', color=colors[i], markersize=10,
           markeredgecolor='black', markeredgewidth=2, zorder=10)

# Initial state
initial_rect = Rectangle(
    (x0_int.lower[0], x0_int.lower[1]),
    x0_int.upper[0] - x0_int.lower[0],
    x0_int.upper[1] - x0_int.lower[1],
    linewidth=2,
    edgecolor='black',
    facecolor='yellow',
    alpha=0.5,
    linestyle='--',
    label='Initial'
)
ax.add_patch(initial_rect)

# Nominal trajectory
ax.plot(x_nom, y_nom, 'k--', linewidth=2, alpha=0.5, label='Nominal path')

ax.set_xlabel('X Position (m)', fontsize=13, fontweight='bold')
ax.set_ylabel('Y Position (m)', fontsize=13, fontweight='bold')
ax.set_title(f'Final Reachable Sets (t={N*dt}s)\nOutput Feedback Controller',
            fontsize=14, fontweight='bold')
ax.legend(fontsize=11, loc='best')
ax.grid(True, alpha=0.3)
ax.axis('equal')

# Right: Overlap bar chart
ax = axes[1]

overlap_labels = list(stats['overlaps'].keys())
overlap_values = list(stats['overlaps'].values())

short_labels = [
    label.replace(' vs ', '\nvs\n').replace(' Fault', '')
    for label in overlap_labels
]

bars = ax.bar(range(len(overlap_values)), overlap_values,
             color=['#9b59b6', '#f39c12', '#16a085'],
             alpha=0.8, edgecolor='black', linewidth=2)

ax.set_xticks(range(len(overlap_labels)))
ax.set_xticklabels(short_labels, fontsize=10)
ax.set_ylabel('Overlap Volume (m²)', fontsize=13, fontweight='bold')
ax.set_title('Pairwise Overlaps', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

# Value labels
for bar, val in zip(bars, overlap_values):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
           f'{val:.5f}',
           ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig('go2_feedback_controller_reachable_sets.pdf', dpi=300, bbox_inches='tight')
plt.show()

print("\n✓ Saved: go2_feedback_controller_reachable_sets.pdf")

## 7. Visualize Feedback Gains Over Time

In [ ]:
# Plot feedback gain evolution
fig, axes = plt.subplots(3, 3, figsize=(15, 10))
fig.suptitle('Feedback Gain Evolution K[t] (3×3 matrix at each timestep)',
            fontsize=16, fontweight='bold')

control_labels = ['vx', 'vy', 'ω']
obs_labels = ['px', 'py', 'θ']

for i in range(3):
    for j in range(3):
        ax = axes[i, j]
        
        # Plot K[t,i,j] over time
        K_ij = K_opt[:, i, j]
        ax.plot(t, K_ij, 'b-', linewidth=2)
        ax.axhline(y=0, color='k', linestyle='--', alpha=0.3)
        
        ax.set_title(f'K[{control_labels[i]}, {obs_labels[j]}]',
                    fontweight='bold')
        ax.set_xlabel('Time (s)')
        ax.set_ylabel('Gain value')
        ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('go2_feedback_gains_evolution.pdf', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Saved: go2_feedback_gains_evolution.pdf")

## 8. Compare with Open-Loop Approach

Let's compare the output feedback controller with a simple open-loop constant input.

In [ ]:
from go2_separating_input_simple import (
    SeparatingInputOptimizer as OpenLoopOptimizer
)

print("Running open-loop optimization for comparison...")

# Use same time horizon as nominal trajectory
openloop_opt = OpenLoopOptimizer(
    scenarios=scenarios,
    x0_int=x0_int,
    dt=dt,
    num_steps=N,
    position_only=True
)

# Optimize constant input
u_openloop_opt, loss_openloop = openloop_opt.optimize(
    u_init=jnp.array([0.5, 0.0, 0.3]),
    learning_rate=0.01,
    num_iters=100,
    verbose=False
)

stats_openloop = openloop_opt.evaluate(u_openloop_opt)

print(f"\n✓ Open-loop optimization complete")
print(f"  Total overlap: {loss_openloop:.6f} m²")

In [ ]:
# Comparison table
print("\n" + "="*70)
print("COMPARISON: Output Feedback vs Open-Loop")
print("="*70)

print(f"\n{'Metric':<40} {'Feedback':>12} {'Open-Loop':>12}")
print("-"*70)
print(f"{'Total overlap (m²)':<40} {stats['total_overlap']:>12.6f} {loss_openloop:>12.6f}")
print()

# Pairwise comparison
for key in stats['overlaps'].keys():
    fb_val = stats['overlaps'][key]
    ol_val = stats_openloop['pairwise_overlaps'][key]
    print(f"{key:<40} {fb_val:>12.6f} {ol_val:>12.6f}")

print()
improvement = (loss_openloop - stats['total_overlap']) / loss_openloop * 100
print(f"Improvement: {improvement:+.2f}%")

if stats['total_overlap'] < loss_openloop:
    print("✓ Feedback controller achieves better separation!")
else:
    print("⚠️ Open-loop performs better (feedback may be over-constrained by tracking)")

print("\nNote: Feedback controller also tracks nominal trajectory,")
print("      while open-loop uses a constant input.")

## 9. Key Insights

### Output Feedback Controller Properties

1. **Dual Objective**: Balances trajectory tracking and fault separation
2. **Time-Varying**: Adapts gains at each timestep for optimal performance
3. **Feedback Structure**: Uses observations to correct for deviations

### Comparison Summary

**Output Feedback**:
- ✅ Tracks desired nominal trajectory
- ✅ Adapts to observations
- ✅ Can balance multiple objectives
- ⚠️ More complex (N×3×obs_dim parameters)
- ⚠️ May compromise separation for tracking

**Open-Loop**:
- ✅ Simple (only 3 parameters)
- ✅ Maximum freedom for separation
- ⚠️ Can't track specific trajectory
- ⚠️ No adaptation to observations

### When to Use Each

- **Use Feedback** when you need to track a specific trajectory while diagnosing faults
- **Use Open-Loop** when pure fault separation is the only goal

### Actuator vs Sensor Faults

Both approaches struggle with **sensor fault separation** because sensor faults don't affect dynamics—only measurements. The robot follows the same physical path regardless of IMU drift.

**Actuator faults** are easily separable because reduced turn rate creates distinctly different trajectories.